# KCS October early-20 semiconductor export source receipt
No SOXX outcomes or model fitting.


In [1]:
import datetime,hashlib,html,json,os,re,shlex,subprocess,sys,tarfile,tempfile
from pathlib import Path
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,_settings,_ssh
card=root/'research/kcs_early20_semiconductor_source_20261023';raw=root/'data/snapshots/kcs_early20_semiconductor_source_20261023';raw.mkdir(parents=True,exist_ok=True)
plan=json.loads((card/'plan.json').read_text());assert plan['frozen'] and not (card/'run_state.json').exists()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
(card/'run_state.json').write_text(json.dumps({'status':'started','plan_sha256':sha(card/'plan.json'),'GETs_started':0},indent=2)+'\n')
receipts=[]
for number,(name,url) in enumerate(plan['source_urls'].items(),start=1):
 body=raw/(name+'.body');headers=raw/(name+'.headers');start=datetime.datetime.now(datetime.timezone.utc).isoformat()
 cmd=['curl','--location','--silent','--show-error','--connect-timeout','5','--max-time','20','--dump-header',str(headers),'--output',str(body),'--write-out','%{http_code}|%{url_effective}',url]
 proc=subprocess.run(cmd,capture_output=True,text=True);end=datetime.datetime.now(datetime.timezone.utc).isoformat();parts=proc.stdout.strip().split('|',1)
 receipt={'source_name':name,'url':url,'started_at_utc':start,'completed_at_utc':end,'curl_exit':proc.returncode,'http_status':parts[0] if parts else None,'effective_url':parts[1] if len(parts)>1 else None,'error':proc.stderr[:300] if proc.returncode else None,'body_path':str(body.relative_to(root)),'body_bytes':body.stat().st_size if body.exists() else 0,'body_sha256':sha(body) if body.exists() else None,'headers_path':str(headers.relative_to(root)),'headers_bytes':headers.stat().st_size if headers.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None}
 receipts.append(receipt);(card/'run_state.json').write_text(json.dumps({'status':'capturing','plan_sha256':sha(card/'plan.json'),'GETs_started':number},indent=2)+'\n')
(card/'response_receipts.json').write_text(json.dumps(receipts,ensure_ascii=False,indent=2)+'\n')
texts={}
for receipt in receipts:
 p=root/receipt['body_path'];texts[receipt['source_name']]=html.unescape(re.sub(r'<[^>]+>',' ',p.read_text(encoding='utf-8',errors='replace'))) if receipt['body_bytes'] and receipt['http_status']=='200' and receipt['source_name']!='KCS_PDF' else ''
normalized={k:re.sub(r'\s+',' ',v) for k,v in texts.items()}
page_checks={k:{'oct_1_20':bool(re.search(r'10\s*월\s*1\s*일.{0,35}10\s*월\s*20\s*일',v)) or ('10.1.' in v and '10.20.' in v),'registered_oct21':'2026.10.21' in v or '2026년 10월 21일' in v,'semiconductor_275_hundred_million_usd':bool(re.search(r'반도체.{0,35}275\s*억\s*달러',v))} for k,v in normalized.items() if k in ('KCS_article','government_portal_mirror')}
pdf=raw/'KCS_PDF.body';pdf_valid=pdf.exists() and pdf.read_bytes().startswith(b'%PDF-')
status='source_receipt_semiconductor_baseline_observed' if pdf_valid and all(all(v.values()) for v in page_checks.values()) and len(page_checks)==2 else 'needs_source_validation'
result={'status':status,'observation_period':'2026-10-01..2026-10-20','KCS_page_registered_date':'2026-10-21','actual_first_receipt_utc':receipts[0]['started_at_utc'],'GETs':len(receipts),'page_checks':page_checks,'KCS_PDF_valid_header':pdf_valid,'semiconductor_amount_hundred_million_USD_visible':275 if all(v.get('semiconductor_275_hundred_million_usd') for v in page_checks.values()) else None,'historical_first_public_bytes_qualified':False,'strategy_forward_day_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'interpretation':'Exact current official source bytes observed after the registered release and before the frozen full-month November capture; this is a source-only prior baseline, not a qualified historical signal or SOXX result.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
entries={str(p.relative_to(root)):sha(p) for p in sorted(raw.iterdir()) if p.is_file()};entries[str((card/'plan.json').relative_to(root))]=sha(card/'plan.json');entries[str((card/'response_receipts.json').relative_to(root))]=sha(card/'response_receipts.json')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='kcs-early20-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for rel in sorted(entries):out.add(root/rel,arcname=rel,recursive=False)
 digest=sha(archive);size=archive.stat().st_size
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(nas_path)).split()[0];assert remote==digest
 with tempfile.TemporaryDirectory(prefix='kcs-early20-readback-') as back:
  fetched=Path(back)/'restore.tar.gz'
  with fetched.open('wb') as output:
   proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'cat '+shlex.quote(nas_path)],stdout=output,stderr=subprocess.PIPE,timeout=90)
  assert proc.returncode==0 and sha(fetched)==digest
  with tarfile.open(fetched,'r:gz') as source:
   members={m.name:m for m in source.getmembers()};assert set(entries)==set(members)
   for rel,expected in entries.items():
    f=source.extractfile(members[rel]);assert f and hashlib.sha256(f.read()).hexdigest()==expected,rel
receipt={'status':'uploaded_exact_source_bundle','archive_sha256':digest,'archive_bytes':size,'entries':entries,'nas_artifact':evidence,'nas_path':nas_path,'remote_hash_verified':True,'scoped_restore_verified':True}
(card/'source_archive_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':status,'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','source_receipts':'response_receipts.json','source_archive_receipt':'source_archive_receipt.json','source_NAS_sha256':digest,'source_NAS_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
(card/'run_state.json').write_text(json.dumps({'status':'completed','plan_sha256':sha(card/'plan.json'),'GETs_completed':len(receipts),'NAS_scoped_restore_verified':True},indent=2)+'\n')
print(json.dumps({'result':result,'archive_sha256':digest},ensure_ascii=False))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"result": {"status": "needs_source_validation", "observation_period": "2026-10-01..2026-10-20", "KCS_page_registered_date": "2026-10-21", "actual_first_receipt_utc": "2026-10-23T00:34:51.196293+00:00", "GETs": 4, "page_checks": {"KCS_article": {"oct_1_20": false, "registered_oct21": false, "semiconductor_275_hundred_million_usd": false}, "government_portal_mirror": {"oct_1_20": true, "registered_oct21": true, "semiconductor_275_hundred_million_usd": true}}, "KCS_PDF_valid_header": false, "semiconductor_amount_hundred_million_USD_visible": null, "historical_first_public_bytes_qualified": false, "strategy_forward_day_qualified": false, "market_outcome_reads": 0, "model_fits": 0, "native_accounts": 0, "interpretation": "Exact current official source bytes observed after the registered release and before the frozen full-month November capture; this is a source-only prior baseline, not a qualified historical signal or SOXX result."}, "archive_sha256": "91a1a3052b13f598e26548fb88a15c95be279

# 韩国海关10月前20日半导体出口：来源抓取结果

事前冻结四个只读官方 URL，各抓取一次，无重试。海关原文和直连 PDF 都返回 **HTTP 400、15 字节**；上一年海关文章因 TLS 超时没有正文。政府政策门户镜像成功返回 **261,318 字节**，SHA-256 `2e325e024bc7cd1c7ed53bd8adca9ef3eda266c11339fd48ecf89bac867926ab`，抓取开始于 **2026-10-23T00:34:55.759267+00:00**。镜像正文明确标出观测区间 2026-10-01 至 10-20、发布日期 10 月 21 日及半导体出口 **275 亿美元**。

这次抓取的综合状态是 `needs_source_validation`：海关原件和 PDF 尚未拿到，不能核对 PDF 细表，也不能把页面登记日期当作 10 月 21 日实际收到的字节。它只为 11 月正式月报之前保留了一个 **10 月 23 日观察到的官方门户前置数值**，不回溯证明历史首发、不计策略前向日、不读取 SOXX 结果。

四个响应或错误体、头部及哈希已保留；NAS 归档 SHA-256 `91a1a3052b13f598e26548fb88a15c95be279f17618a74f129715aab066c4db0`，远端哈希和逐对象限定恢复核验通过。**0 拟合、0 账户、0 订单**。下一步只能按新的、单独冻结的官方门户 PDF 传输修复再试，不重跑本版 URL；即使得到 PDF，仍须核对其内容与发行时点，方可考虑与月底正式原件的来源比较。
